In [ ]:
from inspect import *
def asyn(f):
    flag = f.__code__.co_flags
    if flag & CO_GENERATOR:
        flag ^= CO_GENERATOR | CO_ASYNC_GENERATOR
    elif not flag & CO_ASYNC_GENERATOR:
        flag |= CO_COROUTINE
    f.__code__ = f.__code__.replace(co_flags=flag)
    return f

In [ ]:
from asyncio import coroutine

In [ ]:
async def f():
    return 3

In [ ]:
def g():
    return 3

In [ ]:
set(dir(f.__code__)) - set(dir(g.__code__))

In [ ]:
f.__code__.co_code

In [ ]:
g.__code__.co_code

In [ ]:
7 ^ 2

In [ ]:
src = """
def f():
    return 3

def g():
    t = f() + 3
    t2 = f() - 6
    return t + t2
"""

In [ ]:
from _ast import *
from ast import parse, unparse
def asyn(a):
    def asyn(a):
        if isinstance(a, FunctionDef):
            a.__class__ = AsyncFunctionDef
        for i in dir(a):
            v = getattr(a, i)
            if getattr(v, '__module__', None) == 'ast':
                if isinstance(v, Call):
                    t = Await(v)
                    t.lineno = v.lineno
                    t.col_offset = v.col_offset
                    setattr(a, i, t)
                asyn(v)
            elif isinstance(v, list):
                for i,j in enumerate(v):
                    if isinstance(j, Call):
                        t = Await(j)
                        t.lineno = j.lineno
                        t.col_offset = j.col_offset
                        v[i] = t
                    asyn(j)
        return a
    a = parse(a)
    asyn(a)
    print(unparse(a))
    exec(compile(a, '', 'exec'), globals())

In [ ]:
async def f(n):
    if n > 1:
        a,b = await gather(f(n-1), f(n-2))
        return a + b
    return 0

In [ ]:
@futureMath
async def febonacci(n):
    if n > 1:
        return await f(n-1) + f(n-2)
    return 0

In [ ]:
async def f():
    return 3

In [ ]:
f().__await__()

In [ ]:
from inspect import isawaitable, iscoroutine, iscoroutinefunction

In [ ]:
isawaitable(f())

In [ ]:
t = f().__await__()
isawaitable(t), iscoroutine(t), iscoroutinefunction(t)

In [ ]:
def F():
    print('f1')
    yield from G()
    print('f2')
    yield from G()
    print('f3')

def G():
    print('g1')
    yield
    print('g2')
    yield
    print('g3')

In [ ]:
f = F()

In [ ]:
next(f)

In [ ]:
next(f())

In [ ]:
def log(f):
    def log(*a):
        print(f'{f.__name__}({", ".join(map(str, a))})')
        return f(*a)
    return log

In [ ]:
def cache(f):
    c = {}
    def cache(*a):
        t = c.get(a)
        if t is None:
            t = c[a] = f(*a)
        return t
    return cache

In [ ]:
@cache
@log
def febonacci(n):
    if n > 1:
        return febonacci(n-1) + febonacci(n-2)
    return 1

In [ ]:
from inspect import isawaitable
from asyncio import gather

async def all(a, b):
    if isawaitable(b):
        a,b = await gather(a, b)
    else:
        a = await a
    return a,b

def amath(f):
    def future(*a, **k):
        return Future(f(*a, **k))
    return future

class AMath:
    def __init__(s, a):
        s.a = a
    def __await__(s):
        return s.a.__await__()
    @amath
    async def __add__(s, a):
        s,a = await all(s, a)
        return s + a
    __radd__ = __add__
    @amath
    async def __sub__(s, a):
        s,a = await all(s, a)
        return s - a
    @amath
    async def _rsub__(s, a):
        s,a = await all(s, a)
        return a - s
    @amath
    async def __mul__(s, a):
        s,a = await all(s, a)
        return s * a
    __rmul__ = __mul__
    @amath
    async def __floordiv__(s, a):
        s,a = await all(s, a)
        return s / a
    @amath
    async def __rfloordiv__(s, a):
        s,a = await all(s, a)
        return a / s

In [ ]:
from sys import setrecursionlimit
setrecursionlimit(20000)

In [ ]:
%time febonacci(2000)

In [ ]:
def acache(f):
    c = {}
    async def cache(*a):
        t = c.get(a, ...)
        if t is ...:
            t = c[a] = await f(*a)
        return t
    return cache

In [ ]:
from asyncio import sleep

@amath
@acache
@log
async def febonacci_endpoint():
    await sleep(1)
    return 1

@amath
@acache
@log
def febonacci(n):
    if n > 1:
        return febonacci(n-1) + febonacci(n-2) + 1
    return febonacci_endpoint()

In [ ]:
await febonacci(200)